## 1. Setup and Data Loading
Imports libraries and loads the protein dataset.

# Transfomer without Embeddings on the bigger dataset

In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import torch.nn as nn
import numpy as np
import math

# Load the dataset
df = pd.read_csv('/home/users/ntu/ktang022/scratch/SC4001_Assignment2/data/2018-06-06-ss.cleaned.csv')

# Ensure there's a 'len' column with sequence lengths
if 'len' not in df.columns:
    df['len'] = df['seq'].str.len()

# Pre-process sequences
df['seq'] = df['seq'].str.replace("*", "X") # Replace non-standard aa
df = df[df['has_nonstd_aa'] == False].reset_index(drop=True)

print(df.head())
df.info()
print(df.tail())

  pdb_id chain_code  seq sst8 sst3  len  has_nonstd_aa
0   1A30          C  EDL  CBC  CEC    3          False
1   1B05          B  KCK  CBC  CEC    3          False
2   1B0H          B  KAK  CBC  CEC    3          False
3   1B1H          B  KFK  CBC  CEC    3          False
4   1B2H          B  KAK  CBC  CEC    3          False
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 386333 entries, 0 to 386332
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   pdb_id         386333 non-null  object
 1   chain_code     386333 non-null  object
 2   seq            386333 non-null  object
 3   sst8           386333 non-null  object
 4   sst3           386333 non-null  object
 5   len            386333 non-null  int64 
 6   has_nonstd_aa  386333 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 18.1+ MB
       pdb_id chain_code                                                seq  \
386328   5NUG          B 

## 2. Create Vocabularies
**MODIFIED:** This cell replaces the ESM loader. We now create a vocabulary for the input amino acid sequences (`seq_vocab`) in addition to the label vocabularies.

In [3]:
# Vocabularies for SST8 and SST3 labels
ss8_vocab = {'H': 0, 'G': 1, 'I': 2, 'E': 3, 'B': 4, 'T': 5, 'S': 6, 'C': 7}
ss3_vocab = {'H': 0, 'E': 1, 'C': 2}

# NEW: Create vocabulary for input amino acid sequences
all_chars = set(''.join(df['seq']))
seq_vocab = {char: i+1 for i, char in enumerate(sorted(list(all_chars)))}
seq_vocab['<pad>'] = 0 # Add padding token
vocab_size = len(seq_vocab)

print(f"Sequence vocab size: {vocab_size}")
print(seq_vocab)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Sequence vocab size: 21
{'A': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'K': 9, 'L': 10, 'M': 11, 'N': 12, 'P': 13, 'Q': 14, 'R': 15, 'S': 16, 'T': 17, 'V': 18, 'W': 19, 'Y': 20, '<pad>': 0}


## 3. Define Dataset and Collate Function
**MODIFIED:** The `ProteinDataset` now returns *tokenized sequences* instead of pre-computed embeddings. We also define a `collate_fn` to handle padding of sequences and labels at the batch level.

In [4]:
class ProteinSequenceDataset(Dataset):
    def __init__(self, sequences, sst8_labels, sst3_labels, seq_vocab, ss8_vocab, ss3_vocab):
        self.sequences = sequences
        self.sst8_labels = sst8_labels
        self.sst3_labels = sst3_labels
        self.seq_vocab = seq_vocab
        self.ss8_vocab = ss8_vocab
        self.ss3_vocab = ss3_vocab

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        ss8 = self.sst8_labels[idx]
        ss3 = self.sst3_labels[idx]
        
        # Tokenize sequence
        seq_tokens = [self.seq_vocab.get(c, 0) for c in seq] # Default to <pad> if char not in vocab
        
        # Tokenize labels
        ss8_tokens = [self.ss8_vocab.get(c, -1) for c in ss8]
        ss3_tokens = [self.ss3_vocab.get(c, -1) for c in ss3]
        
        # Ensure label length matches sequence length
        ss8_tokens = ss8_tokens[:len(seq_tokens)]
        ss3_tokens = ss3_tokens[:len(seq_tokens)]
        
        return torch.tensor(seq_tokens, dtype=torch.long), torch.tensor(ss8_tokens, dtype=torch.long), torch.tensor(ss3_tokens, dtype=torch.long)

def collate_fn(batch):
    seqs, ss8s, ss3s = zip(*batch)
    
    # Pad sequences
    padded_seqs = pad_sequence(seqs, batch_first=True, padding_value=seq_vocab['<pad>'])
    
    # Pad labels (use -1 for padding, as in original code)
    padded_ss8s = pad_sequence(ss8s, batch_first=True, padding_value=-1)
    padded_ss3s = pad_sequence(ss3s, batch_first=True, padding_value=-1)
    
    return padded_seqs, padded_ss8s, padded_ss3s

## 4. Split Data and Create Dataloaders
**MODIFIED:** We split the dataframe indices and create the new `ProteinSequenceDataset`. The `DataLoader` now uses our custom `collate_fn`.

In [5]:
# Split indices (same as before)
train_indices, temp_indices = train_test_split(range(len(df)), test_size=0.2, random_state=42)
val_indices, test_indices = train_test_split(temp_indices, test_size=0.5, random_state=42)

# Create datasets
train_dataset = ProteinSequenceDataset(
    df.iloc[train_indices]['seq'].tolist(),
    df.iloc[train_indices]['sst8'].tolist(),
    df.iloc[train_indices]['sst3'].tolist(),
    seq_vocab, ss8_vocab, ss3_vocab
)
val_dataset = ProteinSequenceDataset(
    df.iloc[val_indices]['seq'].tolist(),
    df.iloc[val_indices]['sst8'].tolist(),
    df.iloc[val_indices]['sst3'].tolist(),
    seq_vocab, ss8_vocab, ss3_vocab
)
test_dataset = ProteinSequenceDataset(
    df.iloc[test_indices]['seq'].tolist(),
    df.iloc[test_indices]['sst8'].tolist(),
    df.iloc[test_indices]['sst3'].tolist(),
    seq_vocab, ss8_vocab, ss3_vocab
)

# Create dataloaders with the collate_fn
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

# NEW: Define embedding dim as a hyperparameter
embedding_dim = 128 

In [6]:
import numpy as np

def get_segments(seq):
    """
    Finds segments of identical consecutive values in a sequence.
    Returns a list of (start_index, end_index, label).
    """
    segments = []
    if len(seq) == 0:
        return segments
    
    start = 0
    for i in range(1, len(seq)):
        if seq[i] != seq[start]:
            segments.append((start, i - 1, seq[start]))
            start = i
    segments.append((start, len(seq) - 1, seq[start]))
    return segments

def calculate_sov_sequence(pred_seq, true_seq, valid_labels):
    """
    Calculates SOV for a single sequence for specified valid labels (e.g., only 'H', 'E', 'C').
    """
    true_segs = [s for s in get_segments(true_seq) if s[2] in valid_labels]
    pred_segs = [s for s in get_segments(pred_seq) if s[2] in valid_labels]
    
    if not true_segs:
        return 0.0, 0.0

    sum_sov = 0.0
    total_N = 0.0

    for s_true in true_segs:
        # s_true is (start, end, label)
        label = s_true[2]
        len_true = s_true[1] - s_true[0] + 1
        total_N += len_true

        # Find all overlapping predicted segments of the SAME label
        overlapping_preds = []
        for s_pred in pred_segs:
            if s_pred[2] == label:
                # Check for overlap
                if max(s_true[0], s_pred[0]) <= min(s_true[1], s_pred[1]):
                    overlapping_preds.append(s_pred)

        if not overlapping_preds:
            continue # No overlapping segments of correct type

        for s_pred in overlapping_preds:
            len_pred = s_pred[1] - s_pred[0] + 1
            minov = min(s_true[1], s_pred[1]) - max(s_true[0], s_pred[0]) + 1
            maxov = max(s_true[1], s_pred[1]) - min(s_true[0], s_pred[0]) + 1
            
            # Delta definition from Zemla '99
            delta_val = min(maxov - minov, minov, int(len_true / 2), int(len_pred / 2))
            
            sum_sov += len_true * (minov + delta_val) / maxov

    return sum_sov, total_N

def compute_sov_batch(logits, labels, vocab, valid_labels_set):
    """
    Computes average SOV for a batch.
    Assumes logits: [Batch, SeqLen, Classes], Labels: [Batch, SeqLen]
    """
    preds = logits.argmax(dim=-1).cpu().numpy()
    labels = labels.cpu().numpy()
    
    total_sov_sum = 0
    total_norm_sum = 0
    
    for i in range(len(labels)):
        # Extract non-padded parts of the sequence
        # Assuming -1 is standard padding in your labeled data for loss, 
        # but using the mask from input sequences is safer if labels also have padding.
        valid_mask = labels[i] != -1 
        
        seq_pred = preds[i][valid_mask]
        seq_true = labels[i][valid_mask]
        
        # Convert token IDs back to characters for segment analysis if needed,
        # or just use IDs directly if valid_labels_set uses IDs.
        # Using IDs is faster.
        
        s_sum, n_sum = calculate_sov_sequence(seq_pred, seq_true, valid_labels_set)
        total_sov_sum += s_sum
        total_norm_sum += n_sum
        
    return 100.0 * (total_sov_sum / total_norm_sum) if total_norm_sum > 0 else 0.0

## 5. Define the Transformer Model
**MODIFIED:** This Transformer now includes its own `nn.Embedding` layer and a `PositionalEncoding` layer. It takes token IDs as input, not pre-computed embeddings.

In [7]:
class PositionalEncoding(nn.Module):
    """Standard Transformer Positional Encoding"""
    def __init__(self, d_model, dropout=0.1, max_len=6000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class ProteinTransformer(nn.Module):
    def __init__(self, vocab_size, input_dim=128, num_heads=8, num_layers=4, ff_dim=512, dropout=0.1):
        super().__init__()
        self.input_dim = input_dim
        self.embedding = nn.Embedding(vocab_size, input_dim, padding_idx=seq_vocab['<pad>'])
        self.pos_encoder = PositionalEncoding(input_dim, dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Two separate classifier heads
        self.q8_head = nn.Linear(input_dim, 8)
        self.q3_head = nn.Linear(input_dim, 3)

    def forward(self, x, mask=None):
        """
        x: [batch_size, seq_len] (token IDs)
        mask: [batch_size, seq_len] (padding mask, True where padded)
        """
        x = self.embedding(x) * math.sqrt(self.input_dim)
        x = self.pos_encoder(x)
        
        x = self.transformer_encoder(x, src_key_padding_mask=mask)
        
        # Per-residue classification
        q8_logits = self.q8_head(x)
        q3_logits = self.q3_head(x)
        
        return q8_logits, q3_logits


## 6. Training Loop
**MODIFIED:** The model is initialized with `vocab_size` and the new `embedding_dim`. The loop now iterates over `seqs, ss8, ss3` and creates the padding mask from the input `seqs`.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm

# --- SOV Helper Functions ---
def get_segments(seq):
    """Finds segments of identical consecutive values."""
    segments = []
    if len(seq) == 0:
        return segments
    start = 0
    for i in range(1, len(seq)):
        if seq[i] != seq[start]:
            segments.append((start, i - 1, seq[start]))
            start = i
    segments.append((start, len(seq) - 1, seq[start]))
    return segments

def calculate_sov_sequence(pred_seq, true_seq, valid_labels):
    """Calculates SOV for a single sequence."""
    true_segs = [s for s in get_segments(true_seq) if s[2] in valid_labels]
    pred_segs = [s for s in get_segments(pred_seq) if s[2] in valid_labels]

    if not true_segs:
        return 0.0, 0.0

    sum_sov = 0.0
    total_N = 0.0

    for s_true in true_segs:
        label = s_true[2]
        len_true = s_true[1] - s_true[0] + 1
        total_N += len_true

        overlapping_preds = []
        for s_pred in pred_segs:
            if s_pred[2] == label:
                if max(s_true[0], s_pred[0]) <= min(s_true[1], s_pred[1]):
                    overlapping_preds.append(s_pred)

        if not overlapping_preds: continue

        for s_pred in overlapping_preds:
            len_pred = s_pred[1] - s_pred[0] + 1
            minov = min(s_true[1], s_pred[1]) - max(s_true[0], s_pred[0]) + 1
            maxov = max(s_true[1], s_pred[1]) - min(s_true[0], s_pred[0]) + 1
            delta_val = min(maxov - minov, minov, int(len_true / 2), int(len_pred / 2))
            sum_sov += len_true * (minov + delta_val) / maxov

    return sum_sov, total_N
# ---------------------------

def compute_accuracy(pred_logits, labels):
    """Per-residue accuracy ignoring -1 padding"""
    preds = pred_logits.argmax(-1)
    mask = labels != -1
    correct = (preds[mask] == labels[mask]).sum().item()
    total = mask.sum().item()
    return correct / total if total > 0 else 0.0

# Initialize the model
model = ProteinTransformer(vocab_size=vocab_size, input_dim=embedding_dim)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
model.to(device)

# Get sets of valid label IDs for SOV calculation
# (Assumes ss3_vocab and ss8_vocab are defined globally as in your previous cells)
q3_valid_ids = set(ss3_vocab.values())
q8_valid_ids = set(ss8_vocab.values())

# Losses and optimizer
criterion_q8 = nn.CrossEntropyLoss(ignore_index=-1)
criterion_q3 = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 100
best_val_acc_q8 = 0.0

# --- Early Stopping Parameters ---
patience = 5
counter = 0
# ---------------------------------

for epoch in range(num_epochs):
    model.train()
    train_loss, train_acc_q8, train_acc_q3 = 0, 0, 0

    for seqs, ss8, ss3 in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
        mask = (seqs == seq_vocab['<pad>'])

        q8_logits, q3_logits = model(seqs, mask)

        loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
        loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
        loss = loss_q8 + 0.5 * loss_q3

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_acc_q8 += compute_accuracy(q8_logits, ss8)
        train_acc_q3 += compute_accuracy(q3_logits, ss3)

    train_loss /= len(train_loader)
    train_acc_q8 /= len(train_loader)
    train_acc_q3 /= len(train_loader)

    # Validation
    model.eval()
    val_loss, val_acc_q8, val_acc_q3 = 0, 0, 0
    
    # SOV accumulators
    val_sov8_sum, val_n8_sum = 0.0, 0.0
    val_sov3_sum, val_n3_sum = 0.0, 0.0

    with torch.no_grad():
        for seqs, ss8, ss3 in val_loader:
            seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
            mask = (seqs == seq_vocab['<pad>'])
            q8_logits, q3_logits = model(seqs, mask)

            # Loss & Accuracy
            loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
            loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
            val_loss += (loss_q8 + 0.5 * loss_q3).item()
            val_acc_q8 += compute_accuracy(q8_logits, ss8)
            val_acc_q3 += compute_accuracy(q3_logits, ss3)

            # --- SOV Calculation ---
            # Move to CPU for complex segment logic
            preds_q8 = q8_logits.argmax(dim=-1).cpu().numpy()
            preds_q3 = q3_logits.argmax(dim=-1).cpu().numpy()
            true_ss8 = ss8.cpu().numpy()
            true_ss3 = ss3.cpu().numpy()

            for i in range(len(seqs)):
                # Get valid mask for this sequence (ignore padding -1)
                valid_mask = true_ss8[i] != -1
                
                # Calculate Q8 SOV for this sequence
                s8, n8 = calculate_sov_sequence(preds_q8[i][valid_mask], true_ss8[i][valid_mask], q8_valid_ids)
                val_sov8_sum += s8
                val_n8_sum += n8
                
                # Calculate Q3 SOV for this sequence
                s3, n3 = calculate_sov_sequence(preds_q3[i][valid_mask], true_ss3[i][valid_mask], q3_valid_ids)
                val_sov3_sum += s3
                val_n3_sum += n3
            # -----------------------

    # Averaging
    val_loss /= len(val_loader)
    val_acc_q8 /= len(val_loader)
    val_acc_q3 /= len(val_loader)
    val_sov8 = 100.0 * val_sov8_sum / val_n8_sum if val_n8_sum > 0 else 0.0
    val_sov3 = 100.0 * val_sov3_sum / val_n3_sum if val_n3_sum > 0 else 0.0

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
    print(f"Train Acc Q8={train_acc_q8:.4f}, Val Acc Q8={val_acc_q8:.4f}, Val SOV8={val_sov8:.2f}")
    print(f"Train Acc Q3={train_acc_q3:.4f}, Val Acc Q3={val_acc_q3:.4f}, Val SOV3={val_sov3:.2f}")

    # --- Early Stopping & Checkpointing Logic ---
    if val_acc_q8 > best_val_acc_q8:
        best_val_acc_q8 = val_acc_q8
        counter = 0
        model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save(model_state, "best_scratch_transformer_model.pt")
        print("Validation accuracy improved. Model saved.")
    else:
        counter += 1
        print(f"EarlyStopping counter: {counter} out of {patience}")
        if counter >= patience:
            print("Early stopping triggered.")
            break

Epoch 1/100: 100%|██████████| 19317/19317 [08:36<00:00, 37.42it/s]
/home/users/ntu/ktang022/.conda/envs/myvenv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1: Train Loss=2.0086, Val Loss=1.9879
Train Acc Q8=0.4023, Val Acc Q8=0.4104, Val SOV8=45.85
Train Acc Q3=0.5290, Val Acc Q3=0.5353, Val SOV3=64.91
Validation accuracy improved. Model saved.


Epoch 2/100: 100%|██████████| 19317/19317 [08:34<00:00, 37.56it/s]


Epoch 2: Train Loss=1.9816, Val Loss=1.9659
Train Acc Q8=0.4126, Val Acc Q8=0.4176, Val SOV8=46.39
Train Acc Q3=0.5376, Val Acc Q3=0.5414, Val SOV3=64.13
Validation accuracy improved. Model saved.


Epoch 3/100: 100%|██████████| 19317/19317 [08:34<00:00, 37.51it/s]


...

Epoch 21/100: 100%|██████████| 19317/19317 [08:34<00:00, 37.52it/s]


Epoch 21: Train Loss=1.5788, Val Loss=1.4615
Train Acc Q8=0.5512, Val Acc Q8=0.5861, Val SOV8=53.00
Train Acc Q3=0.6733, Val Acc Q3=0.7104, Val SOV3=71.54
Validation accuracy improved. Model saved.


Epoch 22/100: 100%|██████████| 19317/19317 [08:35<00:00, 37.49it/s]


Epoch 22: Train Loss=1.5628, Val Loss=1.4432
Train Acc Q8=0.5559, Val Acc Q8=0.5911, Val SOV8=53.27
Train Acc Q3=0.6786, Val Acc Q3=0.7154, Val SOV3=71.93
Validation accuracy improved. Model saved.


Epoch 23/100:  10%|█         | 1965/19317 [00:52<07:57, 36.37it/s]

## 7. Final Evaluation on Test Set
**MODIFIED:** Loads the new model and evaluates using the sequence-based `test_loader`.

In [ ]:
# Initialize a new model instance
model = ProteinTransformer(vocab_size=vocab_size, input_dim=embedding_dim)
# Load the best model state
model.load_state_dict(torch.load("best_scratch_transformer_model.pt"))

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model.to(device)
model.eval()

test_loss, test_acc_q8, test_acc_q3 = 0, 0, 0
with torch.no_grad():
    for seqs, ss8, ss3 in test_loader:
        seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
        mask = (seqs == seq_vocab['<pad>'])
        q8_logits, q3_logits = model(seqs, mask)
        
        loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
        loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
        loss = loss_q8 + 0.5 * loss_q3
        
        test_loss += loss.item()
        test_acc_q8 += compute_accuracy(q8_logits, ss8)
        test_acc_q3 += compute_accuracy(q3_logits, ss3)

test_loss /= len(test_loader)
test_acc_q8 /= len(test_loader)
test_acc_q3 /= len(test_loader)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy Q8: {test_acc_q8:.4f}")
print(f"Test Accuracy Q3: {test_acc_q3:.4f}")